# StockEx Clearing House — LLM Fine-Tuning

Fine-tunes a Qwen2.5 Instruct model with QLoRA to act as a clearing house trading agent.

**Runs on both Kaggle and Colab.** Auto-selects model size based on available VRAM:

| GPU | VRAM | Model |
|-----|------|-------|
| T4 (free) | 15 GB | Qwen2.5-7B-Instruct |
| A100 40 GB | 40 GB | Qwen2.5-14B-Instruct |
| A100 80 GB | 80 GB | Qwen2.5-32B-Instruct |

**Resumable training:**
- **Kaggle**: checkpoints saved/restored from dataset `xabonum/stockex-ch-checkpoints`
- **Colab**: checkpoints saved/restored from Google Drive

**Output model:** `RayMelius/stockex-ch-trader` on HuggingFace Hub

**Required secret:** Add `HF_TOKEN` in Kaggle Secrets or Colab Secrets (🔑 icon in left sidebar)

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
# Reinstall bitsandbytes with proper CUDA support (fixes triton.ops error on Colab)
!pip install -q -U bitsandbytes
!pip install -q \
    "transformers>=4.46.3" \
    "peft>=0.13.2" \
    "trl>=0.12.1" \
    "datasets>=3.1.0" \
    "accelerate>=1.1.1" \
    huggingface_hub
print("Dependencies installed.")

In [ ]:
import os, json, random, torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Auto-select model based on available VRAM ─────────────────────────────────
import torch

assert torch.cuda.is_available(), "No GPU found — change runtime to GPU."
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB")

if vram_gb >= 70:
    BASE_MODEL = "Qwen/Qwen2.5-32B-Instruct"
    BATCH_SIZE, GRAD_ACCUM, LR = 1, 16, 1e-4
elif vram_gb >= 35:
    BASE_MODEL = "Qwen/Qwen2.5-14B-Instruct"
    BATCH_SIZE, GRAD_ACCUM, LR = 2, 8,  1e-4
else:                                          # T4 / 15 GB
    BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
    BATCH_SIZE, GRAD_ACCUM, LR = 4, 4,  2e-4

print(f"Selected model: {BASE_MODEL}")

# ── Fixed config ───────────────────────────────────────────────────────────────
OUTPUT_REPO  = "RayMelius/stockex-ch-trader"
OUTPUT_DIR   = "./stockex-ch-trader"
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
NUM_EPOCHS   = 3
MAX_SEQ_LEN  = 512
DATASET_SIZE = 2500

# ── HuggingFace login ─────────────────────────────────────────────────────────
import os

HF_TOKEN = None

# Kaggle
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except:
    pass

# Colab
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
        print("HF_TOKEN loaded from Colab Secrets")
    except:
        pass

# Env fallback
if not HF_TOKEN:
    HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found.")

from huggingface_hub import login
login(token=HF_TOKEN)

## 1. Checkpoint Detection (Resumable Training)

Automatically detects and resumes from the latest checkpoint:
- **Kaggle**: Downloads checkpoints from dataset `xabonum/stockex-ch-checkpoints`
- **Colab**: Restores checkpoints from Google Drive

Only the **last 3 checkpoints** are kept in persistent storage.

In [ ]:
import shutil, math, glob
from transformers.trainer_utils import get_last_checkpoint

# ── Detect environment ─────────────────────────────────────────────
RUNNING_ON_KAGGLE = os.path.exists("/kaggle/working")
RUNNING_ON_COLAB  = os.path.exists("/content") and not RUNNING_ON_KAGGLE

USE_DRIVE = False
DRIVE_CKPT_DIR = None

if RUNNING_ON_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_CKPT_DIR = "/content/drive/MyDrive/stockex-ch-checkpoints"
        USE_DRIVE = True
        print(f"Colab: checkpoints will use {DRIVE_CKPT_DIR}")
    except Exception as e:
        print(f"Colab drive mount failed: {e} — saving locally only")

elif RUNNING_ON_KAGGLE:
    DRIVE_CKPT_DIR = "/kaggle/working/stockex-ch-checkpoints"
    USE_DRIVE = True
    os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

    # Download checkpoint dataset if available
    try:
        import subprocess
        result = subprocess.run(
            ["kaggle", "datasets", "download", "-d", "xabonum/stockex-ch-checkpoints",
             "-p", DRIVE_CKPT_DIR, "--unzip"],
            capture_output=True, text=True, timeout=120
        )
        if result.returncode == 0:
            print(f"Kaggle: downloaded checkpoint dataset -> {DRIVE_CKPT_DIR}")
        else:
            print(f"Kaggle: no existing checkpoint dataset (starting fresh)")
            print(f"  stderr: {result.stderr.strip()}")
    except Exception as e:
        print(f"Kaggle: checkpoint download failed: {e}")

    print(f"Kaggle: checkpoints will use {DRIVE_CKPT_DIR}")

else:
    print("Unknown environment — saving locally only")

# ── Free GPU memory to prevent OOM on resume ────────────────────────
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.ipc_collect()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128,garbage_collection_threshold:0.6"

# ── Restore checkpoint from persistent storage to local output dir ──
os.makedirs(OUTPUT_DIR, exist_ok=True)
if USE_DRIVE:
    os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
    drive_ckpt = get_last_checkpoint(DRIVE_CKPT_DIR)
    if drive_ckpt:
        local_ckpt_name = os.path.basename(drive_ckpt)
        local_ckpt_path = os.path.join(OUTPUT_DIR, local_ckpt_name)
        if not os.path.exists(local_ckpt_path):
            print(f"Restoring checkpoint: {drive_ckpt} -> {local_ckpt_path}")
            shutil.copytree(drive_ckpt, local_ckpt_path)

# ── Detect latest local checkpoint ────────────────────────────────
RESUME_FROM = get_last_checkpoint(OUTPUT_DIR)
SAVE_STEPS = 10

# ── Resume summary ─────────────────────────────────────────────────
train_size      = int(DATASET_SIZE * 0.9)
steps_per_epoch = math.ceil(train_size / (BATCH_SIZE * GRAD_ACCUM))
total_steps     = steps_per_epoch * NUM_EPOCHS

if RESUME_FROM:
    completed_steps = int(os.path.basename(RESUME_FROM).split("-")[-1])
    remaining    = max(0, total_steps - completed_steps)
    pct_done     = 100 * completed_steps / total_steps
    epoch_done   = completed_steps / steps_per_epoch

    print("\n" + "=" * 55)
    print("  RESUMING FROM CHECKPOINT")
    print("=" * 55)
    print(f"  Checkpoint   : {os.path.basename(RESUME_FROM)}")
    print(f"  Steps done   : {completed_steps:,} / {total_steps:,}  ({pct_done:.1f}%)")
    print(f"  Steps left   : {remaining:,}")
    print(f"  Epoch        : {epoch_done:.2f} / {NUM_EPOCHS}")
    print(f"  Epochs left  : {NUM_EPOCHS - epoch_done:.2f}")
    print(f"  Steps/epoch  : {steps_per_epoch:,}")
    print(f"  Save every   : {SAVE_STEPS} steps")
    print("=" * 55 + "\n")
else:
    print("\n" + "=" * 55)
    print("  STARTING FRESH")
    print("=" * 55)
    print(f"  Total steps  : {total_steps:,}")
    print(f"  Steps/epoch  : {steps_per_epoch:,}")
    print(f"  Epochs       : {NUM_EPOCHS}")
    print(f"  Save every   : {SAVE_STEPS} steps")
    print("=" * 55 + "\n")

## 2. Synthetic Dataset Generation

Each training example is a realistic clearing house trading scenario:
- Member state: capital, holdings, obligation remaining
- Market: BBO for each security
- Target: a valid JSON trading decision that respects all constraints

In [ ]:
# Securities from shared_data/securities.txt  (symbol, start_price, current_price)
SECURITIES = [
    {"symbol": "ALPHA", "base": 5.65},
    {"symbol": "PEIR",  "base": 8.35},
    {"symbol": "EXAE",  "base": 6.90},
    {"symbol": "QUEST", "base": 13.35},
    {"symbol": "NBG",   "base": 8.00},
    {"symbol": "EUROB", "base": 3.45},
    {"symbol": "AEG",   "base": 4.75},
    {"symbol": "INTKA", "base": 7.35},
    {"symbol": "AAAK",  "base": 2.75},
    {"symbol": "ATTIK", "base": 4.90},
]

STARTING_CAPITAL = 100_000.0
DAILY_OBLIGATION = 10


def gen_bbo(base_price: float) -> dict:
    """Generate a realistic bid/ask spread around a base price."""
    drift    = random.uniform(-0.05, 0.05)
    mid      = round(base_price * (1 + drift), 2)
    spread   = round(random.choice([0.05, 0.10, 0.15]), 2)
    best_bid = round(mid - spread / 2, 2)
    best_ask = round(mid + spread / 2, 2)
    return {"best_bid": best_bid, "best_ask": best_ask, "mid": mid}


def gen_holdings(bbos: dict) -> list:
    """Randomly generate some holdings for a member."""
    holdings = []
    n = random.randint(0, 4)
    for sym in random.sample(list(bbos.keys()), min(n, len(bbos))):
        qty = random.randint(50, 500)
        mid = bbos[sym]["mid"]
        avg_cost = round(mid * random.uniform(0.92, 1.08), 2)
        holdings.append({"symbol": sym, "quantity": qty, "avg_cost": avg_cost})
    return holdings


def build_prompt(member_id: str, capital: float, holdings: list,
                 obligation_remaining: int, bbos: dict) -> str:
    market_lines = [
        f"  {sym}: Bid {bbo['best_bid']:.2f} / Ask {bbo['best_ask']:.2f}"
        for sym, bbo in sorted(bbos.items())
    ]
    holding_lines = (
        [f"  {h['symbol']}: {h['quantity']} shares @ avg cost {h['avg_cost']:.2f}"
         for h in holdings]
        if holdings else ["  None"]
    )
    return (
        f"You are simulating clearing house member {member_id} making ONE trading decision.\n\n"
        f"Member state:\n"
        f"  Available capital: EUR {capital:,.2f}\n"
        f"  Securities obligation remaining today: {obligation_remaining} more to trade\n"
        f"  Current holdings:\n" + "\n".join(holding_lines) + "\n\n"
        f"Current market (Bid/Ask):\n" + "\n".join(market_lines) + "\n\n"
        f"Rules:\n"
        f"- Do not spend more than your available capital\n"
        f"- Do not sell more shares than you hold\n"
        f"- If you have no holdings, you must BUY\n"
        f"- Choose a realistic price close to the BBO mid-price\n"
        f"- Quantity should be between 10 and 200\n\n"
        f"Respond ONLY with valid JSON, no other text:\n"
        f'Example: {{"symbol": "ALPHA", "side": "BUY", "quantity": 50, "price": 5.65}}'
    )


def gen_decision(capital: float, holdings: list, bbos: dict) -> dict:
    """Generate a rule-valid trading decision for the given state."""
    holdings_value = sum(
        h["quantity"] * bbos.get(h["symbol"], {}).get("mid", h["avg_cost"])
        for h in holdings
    )
    net_worth = capital + holdings_value
    holdings_ratio = holdings_value / net_worth if net_worth > 0 else 0

    if not holdings:
        side = "BUY"
    elif holdings_ratio > 0.6:
        side = random.choices(["SELL", "BUY"], weights=[0.7, 0.3])[0]
    else:
        side = random.choices(["BUY", "SELL"], weights=[0.55, 0.45])[0]

    if side == "BUY":
        affordable = [sym for sym, bbo in bbos.items() if 10 * bbo["best_ask"] <= capital]
        if not affordable:
            sym = min(bbos, key=lambda s: bbos[s]["best_ask"])
        else:
            held_syms = [h["symbol"] for h in holdings]
            weights = [3 if s in held_syms else 1 for s in affordable]
            sym = random.choices(affordable, weights=weights)[0]
        ask = bbos[sym]["best_ask"]
        max_qty = min(200, int(capital / ask))
        qty = random.randint(10, max(10, max_qty))
        price = round(bbos[sym]["mid"] + random.uniform(-0.05, 0.05), 2)
        price = max(bbos[sym]["best_bid"], min(price, ask))
        return {"symbol": sym, "side": "BUY", "quantity": qty, "price": round(price, 2)}
    else:
        h = random.choice(holdings)
        sym = h["symbol"]
        bbo = bbos[sym]
        qty = random.randint(10, min(200, h["quantity"]))
        price = round(bbo["mid"] + random.uniform(-0.05, 0.05), 2)
        price = max(bbo["best_bid"] - 0.05, min(price, bbo["best_ask"]))
        return {"symbol": sym, "side": "SELL", "quantity": qty, "price": round(price, 2)}


def generate_dataset(n: int) -> list:
    examples = []
    member_ids = [f"USR{i:02d}" for i in range(1, 11)]
    scenarios = [
        ((80_000, 100_000), (5, 10), "fresh_member"),
        ((50_000, 80_000),  (0, 5),  "active_member"),
        ((20_000, 50_000),  (0, 2),  "low_capital"),
        ((5_000,  20_000),  (0, 10), "very_low_capital"),
        ((90_000, 100_000), (10, 10),"start_of_day"),
    ]
    for _ in range(n):
        cap_range, obl_range, _ = random.choice(scenarios)
        capital    = round(random.uniform(*cap_range), 2)
        obligation = random.randint(*obl_range)
        member_id  = random.choice(member_ids)
        bbos       = {s["symbol"]: gen_bbo(s["base"]) for s in SECURITIES}
        holdings   = gen_holdings(bbos)
        holdings_cost = sum(h["quantity"] * h["avg_cost"] for h in holdings)
        if holdings_cost > STARTING_CAPITAL - capital:
            scale = (STARTING_CAPITAL - capital) / max(holdings_cost, 1)
            for h in holdings:
                h["quantity"] = max(10, int(h["quantity"] * scale))
        prompt   = build_prompt(member_id, capital, holdings, obligation, bbos)
        decision = gen_decision(capital, holdings, bbos)
        examples.append({"prompt": prompt, "completion": json.dumps(decision)})
    return examples


print(f"Generating {DATASET_SIZE} training examples...")
raw_data = generate_dataset(DATASET_SIZE)
print(f"Done. Example:")
print("PROMPT:\n", raw_data[0]["prompt"])
print("\nCOMPLETION:", raw_data[0]["completion"])

In [ ]:
# Train/val split (90/10)
random.shuffle(raw_data)
split = int(len(raw_data) * 0.9)
train_data = raw_data[:split]
val_data   = raw_data[split:]

train_dataset = Dataset.from_list(train_data)
val_dataset   = Dataset.from_list(val_data)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

## 3. Load Base Model (4-bit QLoRA)

In [ ]:
print(f"Loading tokenizer: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token        = tokenizer.eos_token
tokenizer.padding_side     = "right"
tokenizer.model_max_length = MAX_SEQ_LEN   # replaces max_seq_length in SFTConfig
print("Tokenizer loaded")

In [ ]:
SYSTEM_PROMPT = (
    "You are a StockEx clearing house trading agent. "
    "Given a member's financial state and live market data, "
    "you output a single valid JSON trading decision that respects all capital and holdings constraints. "
    "Never output anything other than the JSON object."
)


def format_chat(example):
    """Apply the model's chat template to produce a training string."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": example["prompt"]},
        {"role": "assistant", "content": example["completion"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


train_dataset = train_dataset.map(format_chat)
val_dataset   = val_dataset.map(format_chat)

print("Sample formatted text:")
print(train_dataset[0]["text"][:600], "...")

In [ ]:
# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading model: {BASE_MODEL} (4-bit)")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
model.config.use_cache = False
model.config.pretraining_tp = 1
print(f"Model loaded. Parameters: {model.num_parameters()/1e9:.2f}B")

from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

## 3b. LoRA Configuration

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable/1e6:.1f}M / {total/1e6:.0f}M ({100*trainable/total:.2f}%)")

## 4. Train

In [ ]:
from transformers import TrainerCallback
import shutil, glob

KAGGLE_DATASET_ID = "xabonum/stockex-ch-checkpoints"
MAX_CHECKPOINTS_PERSISTENT = 3  # keep only last 3 in Kaggle dataset / Drive

# ── Create Kaggle dataset metadata if needed ────────────────────────
if USE_DRIVE and DRIVE_CKPT_DIR:
    metadata_path = os.path.join(DRIVE_CKPT_DIR, "dataset-metadata.json")
    if not os.path.exists(metadata_path):
        metadata = {
            "title": "stockex-ch-checkpoints",
            "id": KAGGLE_DATASET_ID,
            "licenses": [{"name": "CC0-1.0"}]
        }
        with open(metadata_path, "w") as f:
            json.dump(metadata, f, indent=2)
        print("Created dataset-metadata.json")


def cleanup_old_checkpoints(ckpt_dir, keep=MAX_CHECKPOINTS_PERSISTENT):
    """Keep only the last `keep` checkpoints in persistent storage."""
    ckpts = sorted(
        glob.glob(os.path.join(ckpt_dir, "checkpoint-*")),
        key=lambda p: int(os.path.basename(p).split("-")[-1])
    )
    while len(ckpts) > keep:
        old = ckpts.pop(0)
        shutil.rmtree(old)
        print(f"[Checkpoint] Removed old: {os.path.basename(old)}")


class CheckpointSyncCallback(TrainerCallback):
    """Copy checkpoint to persistent storage, push LoRA to HF Hub, upload to Kaggle dataset."""

    def on_save(self, args, state, control, **kwargs):
        ckpt_dir = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
        if not os.path.isdir(ckpt_dir):
            return

        # 1. Save to persistent folder (Drive / Kaggle working dir)
        if USE_DRIVE and DRIVE_CKPT_DIR:
            dest = os.path.join(DRIVE_CKPT_DIR, f"checkpoint-{state.global_step}")
            os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
            try:
                shutil.copytree(ckpt_dir, dest, dirs_exist_ok=True)
                print(f"[Checkpoint] Saved -> {dest}")
            except Exception as e:
                print(f"[Checkpoint] Copy failed: {e}")

            # Cleanup: keep only last 3 checkpoints in persistent storage
            try:
                cleanup_old_checkpoints(DRIVE_CKPT_DIR)
            except Exception as e:
                print(f"[Checkpoint] Cleanup failed: {e}")

        # 2. Push LoRA adapter to HF Hub
        try:
            kwargs["model"].push_to_hub(
                OUTPUT_REPO,
                commit_message=f"Checkpoint step {state.global_step} (epoch {state.epoch:.2f})",
                token=HF_TOKEN,
            )
            print(f"[Checkpoint] Pushed step {state.global_step} -> HF Hub")
        except Exception as e:
            print(f"[Checkpoint] HF push failed: {e}")

        # 3. Upload to Kaggle dataset (Kaggle only)
        if RUNNING_ON_KAGGLE and USE_DRIVE:
            try:
                os.system(
                    f"kaggle datasets version -p {DRIVE_CKPT_DIR} "
                    f"-m 'Checkpoint step {state.global_step}' --dir-mode zip"
                )
                print(f"[Checkpoint] Kaggle dataset updated for step {state.global_step}")
            except Exception as e:
                print(f"[Checkpoint] Kaggle dataset update failed: {e}")


sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    report_to="none",
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    callbacks=[CheckpointSyncCallback()],
)

if RESUME_FROM:
    print(f"Resuming from checkpoint: {RESUME_FROM}")
else:
    print(f"Starting training from scratch. Save every {SAVE_STEPS} steps.")

trainer.train(resume_from_checkpoint=RESUME_FROM)
print("Training complete.")

# ── Immediately save adapter so subsequent cells don't depend on trainer state ──
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

In [ ]:
import matplotlib.pyplot as plt

# Extract losses from trainer log history
train_steps, train_loss = [], []
eval_steps, eval_loss = [], []

for entry in trainer.state.log_history:
    if "loss" in entry and "eval_loss" not in entry:
        train_steps.append(entry["step"])
        train_loss.append(entry["loss"])
    if "eval_loss" in entry:
        eval_steps.append(entry["step"])
        eval_loss.append(entry["eval_loss"])

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_steps, train_loss, label="Train Loss", alpha=0.8)
ax.plot(eval_steps, eval_loss, label="Eval Loss", marker="o", markersize=4)
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final train loss: {train_loss[-1]:.4f}")
print(f"Final eval loss:  {eval_loss[-1]:.4f}")

## 5. Save & Push to HuggingFace Hub

Merges LoRA adapters into the base model weights and pushes the full model.

In [ ]:
from peft import PeftModel

# Reload base model in fp16 for merging (can't merge with 4-bit)
print("Reloading base model in fp16 for adapter merge...")
del model
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
merged_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
merged_model = merged_model.merge_and_unload()
print("Adapters merged.")

In [ ]:
print(f"Pushing merged model to: {OUTPUT_REPO}")
merged_model.push_to_hub(
    OUTPUT_REPO,
    token=HF_TOKEN,
    commit_message="StockEx CH Trader: QLoRA fine-tuned Qwen2.5-32B-Instruct",
)
tokenizer.push_to_hub(
    OUTPUT_REPO,
    token=HF_TOKEN,
    commit_message="Tokenizer for StockEx CH Trader (Qwen2.5-32B-Instruct base)",
)
print(f"✓ Model pushed to https://huggingface.co/{OUTPUT_REPO}")

## 6. Inference Test

Verify the model generates valid JSON trading decisions.

In [ ]:
import re
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    device_map="auto",
)

test_cases = [
    {
        "desc": "New member, no holdings, must trade",
        "capital": 100_000.0,
        "holdings": [],
        "obligation": 10,
    },
    {
        "desc": "Experienced member with holdings, low obligation",
        "capital": 65_000.0,
        "holdings": [
            {"symbol": "ALPHA", "quantity": 300, "avg_cost": 5.60},
            {"symbol": "QUEST", "quantity": 150, "avg_cost": 13.20},
        ],
        "obligation": 2,
    },
    {
        "desc": "Low capital, large holdings",
        "capital": 8_000.0,
        "holdings": [
            {"symbol": "PEIR",  "quantity": 500, "avg_cost": 8.30},
            {"symbol": "NBG",   "quantity": 200, "avg_cost": 7.95},
        ],
        "obligation": 5,
    },
]

test_bbos = {s["symbol"]: gen_bbo(s["base"]) for s in SECURITIES}

print("=" * 70)
for tc in test_cases:
    print(f"\nSCENARIO: {tc['desc']}")
    prompt = build_prompt("USR01", tc["capital"], tc["holdings"], tc["obligation"], test_bbos)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    output = pipe(
        messages,
        max_new_tokens=60,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = output[0]["generated_text"][-1]["content"].strip()
    print(f"RESPONSE: {response}")
    try:
        m = re.search(r"\{[^}]+\}", response)
        if m:
            d = json.loads(m.group())
            assert d["side"] in ("BUY", "SELL")
            assert d["symbol"] in [s["symbol"] for s in SECURITIES]
            assert d["quantity"] > 0
            assert d["price"] > 0
            print(f"✓ Valid JSON: {d}")
        else:
            print("✗ No JSON found in response")
    except Exception as e:
        print(f"✗ Invalid: {e}")
    print("-" * 70)

## 7. Activate in StockEx

The clearing house already uses `RayMelius/stockex-ch-trader` as default.

To switch to this model in a running StockEx instance:

**HuggingFace Spaces** — add to secrets:
```
HF_MODEL = RayMelius/stockex-ch-trader
HF_TOKEN = <your token>
```

**Docker Compose** — already set in `docker-compose.yml`:
```yaml
environment:
  - HF_MODEL=RayMelius/stockex-ch-trader
  - HF_TOKEN=<your token>
```